# DVD Video Depth on CUDA T4 (Colab)

Full-GPU inference with [`generate_depth_cuda.py`](../generate_depth_cuda.py): DiT + VAE resident on the T4.
Google Drive is used **only** for the input video and the output depth MP4.

**Runtime:** Runtime → Change runtime type → GPU (T4).

## 1. Clone the repo

In [ ]:
# Set your GitHub URL after pushing this repo, then run.
REPO_URL = "https://github.com/YOUR_USER/dvd.git"  # <-- edit me
REPO_DIR = "/content/dvd"

import os
from pathlib import Path

if Path(REPO_DIR).exists():
    print(f"Already cloned at {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("cwd:", os.getcwd())

## 2. Install pip and system dependencies

In [ ]:
# ffmpeg for OpenCV decode/encode; project deps; vendored DVD (no CUDA extras).
# Colab already ships a CUDA-enabled torch — keep it unless pip upgrades clash.
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null
!pip install -q -r requirements.txt
!pip install -q -e vendor/DVD --no-deps
print("Install done.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Folder on Drive that holds your input video (edit these paths).
DRIVE_DIR = Path("/content/drive/MyDrive/dvd_videos")  # <-- edit me
INPUT_VIDEO = DRIVE_DIR / "input.mp4"  # <-- edit me

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("DRIVE_DIR:", DRIVE_DIR)
print("INPUT_VIDEO exists:", INPUT_VIDEO.exists(), "->", INPUT_VIDEO)

## 4. Check runtime

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → T4)."
idx = torch.cuda.current_device()
name = torch.cuda.get_device_name(idx)
props = torch.cuda.get_device_properties(idx)
free, total = torch.cuda.mem_get_info(idx)
print(f"GPU: {name}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB total, "
      f"{free / 1024**3:.1f}/{total / 1024**3:.1f} GiB free")
print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if "T4" not in name:
    print(f"Note: expected a T4; got {name!r}. Full-GPU path should still work if VRAM ≥ ~15 GiB.")

## 5. Pull weights and models

In [ ]:
import os
from pathlib import Path

# Optional: paste a HF token if the DVD repo requires auth.
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # or set: HF_TOKEN = "hf_..."
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

ckpt = Path("ckpt/dvd_1.1.safetensors")
cfg = Path("ckpt/model_config.yaml")
repo_cfg = Path("configs/model_config.yaml")

if not ckpt.exists():
    print("Downloading DVD weights into ./ckpt/ …")
    !python scripts/download_weights.py
else:
    print("Checkpoint already present:", ckpt.resolve())

# Config is small and also shipped at configs/model_config.yaml in the repo.
if not cfg.exists() and repo_cfg.exists():
    Path("ckpt").mkdir(parents=True, exist_ok=True)
    cfg.write_text(repo_cfg.read_text())
    print("Copied configs/model_config.yaml ->", cfg.resolve())

assert ckpt.exists(), f"Missing {ckpt} — re-run download with a valid HF_TOKEN if needed."
assert cfg.exists() or repo_cfg.exists(), (
    "Missing model_config.yaml under ckpt/ or configs/."
)
print("Weights OK:", ckpt, "| config:", cfg if cfg.exists() else repo_cfg)
print("Wan2.1 backbone downloads into ./models/ on first model load (next cells).")

## 6. Small test

In [ ]:
from pathlib import Path

# Smoke test needs demo video + weights from cell 5.
demo = Path("vendor/DVD/demo/robot_navi.mp4")
assert demo.exists(), f"Missing demo video: {demo}"
assert Path("ckpt/dvd_1.1.safetensors").exists(), (
    "Missing ckpt/dvd_1.1.safetensors — re-run cell 5 (Pull weights)."
)

!python generate_depth_cuda.py \
  --input-video vendor/DVD/demo/robot_navi.mp4 \
  --output-dir outputs/smoke \
  --cache-dir .cache \
  --max-frames 17 \
  --height 256 \
  --width 448 \
  --window-size 9 \
  --overlap 3 \
  --no-upsample

outs = list(Path("outputs/smoke").glob("*_depth_gray.mp4"))
print("Smoke outputs:", outs)
assert outs, "Smoke test produced no depth video"

## 7. Run on Drive input → save output next to it

In [ ]:
from pathlib import Path

assert INPUT_VIDEO.exists(), f"Put your video at {INPUT_VIDEO} (or edit INPUT_VIDEO above)."

# Output lands in the same Drive folder as `<stem>_depth_gray.mp4`.
!python generate_depth_cuda.py \
  --input-video "{INPUT_VIDEO}" \
  --output-dir "{DRIVE_DIR}" \
  --cache-dir .cache

out = DRIVE_DIR / f"{INPUT_VIDEO.stem}_depth_gray.mp4"
print("Expected output:", out)
print("Exists:", out.exists())
assert out.exists(), f"Missing output at {out}"